# Mission 2 L1 재학습 노트북 (보험 제출용 초안 · main `4bfd2c4` · CPU 전용)

- 용도: L2 재학습이 10-14 전에 끝나지 못할 때 M2 제출본(`M2-L1-9k`, 공식 Val 84.26%)
- 모델: 발화 구간 음향 통계 53차원 + LightGBM 기본값, Training(서울) 9,000통화 샘플(`--n-calls 9000 --seed 42`)
- 에폭/조기종료 없음. 내부 검증은 call_id 그룹 80/20(`--seed 42`). **공식 Validation은 마지막 평가만**
- 입력 규칙(#41): 대상 발화 startAt~endAt 음성만. text·발화 순서·통화 단위 정보 미사용. 제공 라벨 그대로
- GPU 불필요. 위에서 아래로 한 번 실행하면 원본 → feature → 학습 → 제출 `inference.py`로 공식 Val → 판정까지 재현됩니다.

## 0. 설정

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, time

COMMIT    = "4bfd2c4"
REPO      = Path("/content/dcc2026-prelim-main-4bfd2c4")
DATA_ROOT = Path("/content/drive/.shortcut-targets-by-id/1Ezl13Cpmt2lGgl-FqISYJWfop8omii6z/대학부 데이터")
MANIFEST  = REPO / "data/manifests/dcc_manifest_v1.csv"   # 32,840행
VAL_AUDIO_DIR = DATA_ROOT / "Validation/<서울 원천 WAV 폴더>"   # 실제 경로로 채울 것 (평평한 폴더면 그대로)
VAL_LABEL_DIR = DATA_ROOT / "Validation/<서울 라벨 JSON 폴더>"

OUT = Path("/content/drive/MyDrive/dcc_runs/m2_l1_9k_rerun_" + COMMIT)  # 처음부터 Drive (끊겨도 진행분 보존)
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "run_log.txt"
N_CALLS, SEED = 9000, 42
REF_VAL_ACC = 0.8425504926438404      # 03:43 KST 실행 결과
REF_CKPT_SHA = "3f1e872a5d1025f0630fb0327dac551711ed56dd40409153c5756c6bd0ef8780"
RUN_BUILD = RUN_TRAIN = RUN_EVAL = True

os.chdir(REPO); sys.path.insert(0, str(REPO / "src"))
os.environ["DCC_DATA_ROOT"] = str(DATA_ROOT)

def run(cmd):
    line = "$ " + " ".join(map(str, cmd)); print(line)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(f"\n[{time.strftime('%F %T')}] {line}\n")
        p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for l in p.stdout: print(l, end="", flush=True); f.write(l)
        rc = p.wait()
    if rc != 0: raise RuntimeError(f"exit {rc}: {line}")

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

## 1. 환경 기록

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "pyarrow", "librosa", "scikit-learn", "lightgbm", "tqdm"])
import numpy, pandas, sklearn, lightgbm, librosa
print("commit", COMMIT, "| numpy", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__,
      "| lightgbm", lightgbm.__version__, "| librosa", librosa.__version__, "| cpus", os.cpu_count())
assert MANIFEST.exists(), "manifest 없음 — Colab에 만들어 둔 dcc_manifest_v1.csv를 복사"

## 2. Feature 추출 (Training 9,000통화 · 200통화마다 Drive에 중간 저장)

In [ ]:
FEAT, IDX, FAIL = OUT/"m2_features_train_9k.parquet", OUT/"m2_utterances_train_9k.parquet", OUT/"m2_build_failures_9k.jsonl"
if RUN_BUILD:
    # 주의: --resume은 --n-calls 0에서만 허용됨. 9k 샘플은 끊기면 처음부터 다시 (중간 저장분으로 진행 위치만 확인)
    run([sys.executable, "scripts/build_mission2_dataset.py", "--manifest", MANIFEST, "--data-root", DATA_ROOT,
         "--n-calls", N_CALLS, "--seed", SEED, "--workers", os.cpu_count(), "--flush-every", 200,
         "--index-output", IDX, "--features-output", FEAT, "--failures-log", FAIL])
import pandas as pd
f = pd.read_parquet(FEAT); print("발화", len(f), "| 통화", f["call_id"].nunique() if "call_id" in f else "?")

## 3. 학습 (LightGBM 기본값 · 내부 call_id 그룹 80/20)

In [ ]:
CKPT, TRAIN_LOG = OUT/"m2_baseline_lgbm_9k.pkl", OUT/"m2_exp_log_9k.json"
if RUN_TRAIN:
    run([sys.executable, "scripts/train_mission2_baseline.py", "--features", FEAT, "--seed", SEED,
         "--test-size", 0.2, "--model-output", CKPT, "--log-output", TRAIN_LOG])
print(json.dumps(json.load(open(TRAIN_LOG)), ensure_ascii=False, indent=1)[:3000])
CKPT_SHA = sha256(CKPT)
print("ckpt sha256", CKPT_SHA, "| 기존 실행과 동일" if CKPT_SHA == REF_CKPT_SHA else "| 기존과 다름(패키지 버전 차이면 참고만)")

## 4. 공식 Validation — 제출용 `inference.py` 그대로 (평가만)

In [ ]:
PRED = OUT/"mission2_val.csv"
if RUN_EVAL:
    run([sys.executable, "scripts/mission2_inference.py", "--audio_dir", VAL_AUDIO_DIR, "--label_dir", VAL_LABEL_DIR,
         "--ckpt_path", CKPT, "--output", PRED])

In [ ]:
# 정답 대조: inference.py는 라벨 파일의 발화 순서대로 행을 씀 → 파일별 순서로 맞춤
import numpy as np
pred = pd.read_csv(PRED)
pred["stem"] = pred["audio_file_name"].map(lambda s: Path(str(s)).stem)
rows = []
for jp in sorted(Path(VAL_LABEL_DIR).rglob("*.json")):
    doc = json.load(open(jp, encoding="utf-8"))
    for i, u in enumerate(doc["utterances"]):      # startAt/endAt/speaker만 읽음 (text 미사용)
        rows.append({"stem": jp.stem, "i": i, "startAt": u["startAt"], "endAt": u["endAt"], "y": int(u["speaker"])})
gt = pd.DataFrame(rows)
pred["i"] = pred.groupby("stem").cumcount()
df = gt.merge(pred[["stem", "i", "speaker"]], on=["stem", "i"], how="left", validate="one_to_one")
assert df["speaker"].notna().all() and len(df) == len(pred), "행 수/정렬 불일치"
df["ok"] = df["y"] == df["speaker"]
acc = df.ok.mean()
cm = pd.crosstab(df.y, df.speaker).reindex(index=[0, 1], columns=[0, 1], fill_value=0).values.tolist()
print(f"공식 Val Acc {acc:.4%} | 발화 {len(df)} | 통화 {df.stem.nunique()} | 혼동행렬 {cm}")

## 5. 슬라이스 (분석용)

In [ ]:
df["dur"] = df["endAt"] - df["startAt"]          # 단위가 초인지 확인 (ms면 /1000)
def call_overlap_ratio(g):
    iv = g[["startAt", "endAt"]].to_numpy(float); iv = iv[iv[:, 0].argsort()]
    total = (iv[:, 1] - iv[:, 0]).sum(); ov = 0.0; end = -np.inf
    for s, e in iv:                                # 통화 안에서 두 발화 이상 겹치는 시간
        if s < end: ov += min(e, end) - s
        end = max(end, e)
    span = iv[:, 1].max() - iv[:, 0].min()
    return ov / span if span > 0 else 0.0
ratio = df.groupby("stem").apply(call_overlap_ratio); p90 = ratio.quantile(0.9)
top = set(ratio[ratio >= p90].index)
S = {"overall": df, f"overlap_top10pct_calls(p90={p90:.4f})": df[df.stem.isin(top)], "overlap_rest": df[~df.stem.isin(top)],
     "dur_<0.5s": df[df.dur < 0.5], "dur_0.5-1s": df[(df.dur >= 0.5) & (df.dur < 1)], "dur_<1s": df[df.dur < 1],
     "dur_>=1s": df[df.dur >= 1], "recall_대원(0)": df[df.y == 0], "recall_신고자(1)": df[df.y == 1]}
slices = {k: {"n": int(len(d)), "acc": float(d.ok.mean())} for k, d in S.items()}
json.dump(slices, open(OUT/"m2_l1_9k_official_val_slices.json", "w"), ensure_ascii=False, indent=2)
pd.DataFrame(slices).T

## 6. 판정 · 환경 고정

In [ ]:
diff = acc - REF_VAL_ACC
verdict = "재현 성공" if abs(diff) <= 0.003 else "±0.3%p 밖 — 원인 확인"
summary = {"candidate": "M2-L1-9k", "commit": COMMIT, "accuracy": acc, "ref_accuracy": REF_VAL_ACC, "diff_pp": diff * 100,
           "verdict": verdict, "n_utt": int(len(df)), "n_calls": int(df.stem.nunique()), "confusion_matrix": cm,
           "ckpt_sha256": CKPT_SHA, "same_ckpt_as_ref": CKPT_SHA == REF_CKPT_SHA}
json.dump(summary, open(OUT/"m2_l1_9k_official_val_summary.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=1))
subprocess.run(f"{sys.executable} -m pip freeze > '{OUT}/pip_freeze.txt'", shell=True)